# Experiment 1: Progressive Disclosure vs Load-Everything

**The silly way:** dump ALL 7 papers into one call to answer one tiny question.
**The smart way:** show the agent a small *index*, let it load only the ONE paper it needs.

We measure both with **real** Bedrock token counts and see the difference.

## Setup

In [ ]:
import json, time, os, sys
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# Find the project root whether we launch from notebooks/ or the repo root,
# and make `core` and `tools` importable.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/home/samir-dahal/context_engineering-01')

In [ ]:
# Which Claude model + region to use (override in .env).
MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "us.anthropic.claude-3-5-haiku-20241022-v1:0")
REGION   = os.getenv("AWS_REGION", "us-east-1")
print(MODEL_ID, "@", REGION)

us.anthropic.claude-3-5-haiku-20241022-v1:0 @ us-east-1


In [ ]:
# The real Bedrock tokenizer (falls back to an estimate if no AWS creds yet).
from core.tokenizer import count_tokens
count_tokens("hello world hii my name is samir dahal ")

[tokenizer] WARNING: AWS CountTokens unavailable -- using the words*1.3 ESTIMATE. Numbers are approximate until AWS creds work.


10

In [ ]:
# Load the index (the "map") and the question cards.
INDEX = json.loads((PROJECT_ROOT / "data" / "index.json").read_text())


EVALS = json.loads((PROJECT_ROOT / "data" / "evals" / "exp01.json").read_text())
DEMO_QUERY = EVALS[0]["question"]

print(f"{len(INDEX)} papers | demo query: {DEMO_QUERY}")

7 papers | demo query: What do attention scores sum to, and why?


## Arm 1 - Naive (load everything)

Glue all 7 papers together and send them in one single Bedrock call.

**Heads up:** all 7 papers in full are ~227,000 tokens, which *overflows*
Claude's 200k context window. That overflow is itself the lesson - load-everything
literally does not fit. So we cap the blob to the most words that fit, and even
capped it dwarfs the smart approach.

In [ ]:
# Glue every paper into one giant blob, and note each paper's real token size.
naive_blob = ""
naive_breakdown = {}
for paper in INDEX:
    text = (PROJECT_ROOT / paper["path"]).read_text(encoding="utf-8", errors="ignore")
    naive_blob += f"\n\n=== {paper['title']} ===\n{text}"
    naive_breakdown[paper["title"]] = count_tokens(text)


naive_breakdown




{'Attention Is All You Need': 7924,
 'Lost in the Middle': 12696,
 'RULER': 17156,
 'Retrieval-Augmented Generation': 12852,
 'A Survey on Hallucination in LLMs': 47554,
 'Indirect Prompt Injection': 23412,
 'RLHF effects on diversity': 22593}

In [ ]:
# How heavy is the full blob? (~2 real tokens per word for dense papers)
print(f"{len(naive_blob.split()):,} words  ~= {len(naive_blob.split()) * 2:,} real tokens")
print("That overflows the 200k window for haiku model, so we cap it next.")

110,951 words  ~= 221,902 real tokens
That overflows the 200k window for haiku model, so we cap it next.


In [ ]:
# Cap to the most words that fit the context window, then build the prompt.
NAIVE_MAX_WORDS = 97500        #
naive_text = " ".join(naive_blob.split()[:NAIVE_MAX_WORDS])
naive_prompt = f"{naive_text}\n\nQuestion: {DEMO_QUERY}\nAnswer:"

import boto3
bedrock = boto3.client("bedrock-runtime", region_name=REGION)
print(f"Capped naive context to {NAIVE_MAX_WORDS:,} words so it fits.")

Capped naive context to 97,500 words so it fits.


In [ ]:
# ONE single Claude call. Time it.
t0 = time.time()
naive_resp = bedrock.converse(
    modelId=MODEL_ID,
    messages=[{"role": "user", "content": [{"text": naive_prompt}]}],
    inferenceConfig={"maxTokens": 1000, "temperature": 0.0},
)
naive_latency = time.time() - t0
print(f"done in {naive_latency:.2f}s")

done in 55.05s


In [ ]:
naive_resp

{'ResponseMetadata': {'RequestId': 'df69ed92-dd78-4377-a2cb-7492e4f6a1e8',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sat, 06 Jun 2026 04:23:35 GMT',
   'content-type': 'application/json',
   'content-length': '1613',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'df69ed92-dd78-4377-a2cb-7492e4f6a1e8'},
  'RetryAttempts': 0},
 'output': {'message': {'role': 'assistant',
   'content': [{'text': "In the Transformer architecture, attention scores sum to 1 for each query position across all key positions. This is achieved through the softmax operation in the attention mechanism. Here's a detailed explanation:\n\n1. Computation of Attention Scores:\n- For each query vector q, dot product scores are computed with all key vectors k\n- These raw dot product scores are scaled by 1/√(d_k), where d_k is the dimension of the key vectors\n\n2. Softmax Normalization:\n- The scaled dot product scores are passed through a softmax function\n- Softmax transforms the raw scores into a pr

In [ ]:
# Pull out the real answer and the real token usage.
naive_answer = naive_resp["output"]["message"]["content"][0]["text"]
naive_usage  = naive_resp["usage"]   # REAL: inputTokens / outputTokens / totalTokens
print(naive_answer[:300], "...")

In the Transformer architecture described in the "Attention Is All You Need" paper, the attention scores for each query sum to 1 due to the softmax operation.

Specifically, in the scaled dot-product attention mechanism (Section 3.2.1), the attention scores are computed as follows:

1. First, dot pr ...


In [ ]:
naive_input_tokens = naive_usage["inputTokens"]
print(f"Naive input tokens: {naive_input_tokens:,}")

Naive input tokens: 200,250


## Arm 2 - Progressive disclosure

The agent sees only a tiny *menu* of papers, then uses `load_document`
to open the ONE it needs.

In [ ]:
# The lightweight index the agent sees - titles + one-line descriptions only.
index_prompt = "Available research papers:\n"
for p in INDEX:
    index_prompt += f"[{p['id']}] {p['title']} - {p['desc']}\n"
print(index_prompt)

Available research papers:
[1706.03762] Attention Is All You Need - transformer architecture; self-attention; attention scores sum to 1.0
[2307.03172] Lost in the Middle - how LLMs use long contexts; U-shaped position performance curve
[2404.06654] RULER - measuring the real effective context length of long-context models
[2005.11401] Retrieval-Augmented Generation - RAG for knowledge-intensive NLP tasks
[2311.05232] A Survey on Hallucination in LLMs - taxonomy of hallucination including factuality vs faithfulness
[2302.12173] Indirect Prompt Injection - compromising LLM apps via malicious instructions in retrieved data
[2310.06452] RLHF effects on diversity - how RLHF reduces output diversity in language models



In [ ]:
# How tiny is the menu vs the giant blob?
print(f"index: ~{count_tokens(index_prompt):,} tokens  (blob was ~{sum(naive_breakdown.values()):,})")

index: ~126 tokens  (blob was ~144,187)


In [ ]:
# The agent's instructions.
system_prompt = f"""You are a research assistant with access to AI/ML papers.
{index_prompt}
When answering a question:
1. Identify which paper contains the answer
2. Call load_document with that paper's ID
3. Read it and answer with a citation

Be precise. Cite the paper ID in your answer."""

In [ ]:
# Build the agent: Bedrock Claude + the load_document "hand".
from strands import Agent
from strands.models import BedrockModel
from tools.load_document import load_document

model = BedrockModel(model_id=MODEL_ID, region_name=REGION,
                     temperature=0.0, max_tokens=1000)
agent = Agent(model=model, tools=[load_document], system_prompt=system_prompt)

In [ ]:
# Run it. The agent picks a paper, loads only that one, then answers.
t0 = time.time()
result = agent(DEMO_QUERY)
prog_latency = time.time() - t0
print(f"\n done in {prog_latency:.2f}s")

Based on the "Attention Is All You Need" paper [1706.03762], attention scores sum to 1.0 due to the softmax normalization. 

Specifically, in the Scaled Dot-Product Attention formula (Equation 1), the softmax function is used to convert raw compatibility scores into a probability distribution:

Attention(Q, K, V) = softmax(QKT / √dk) V

The softmax function ensures that:
1. All attention scores for a given query sum to 1.0
2. Scores are non-negative
3. Each position can flexibly allocate its attention across different input positions

The authors provide two key reasons for this approach:
1. It converts raw dot-product compatibility scores into a proper probability distribution
2. For large dimension sizes (dk), it prevents the softmax from entering regions with extremely small gradients by scaling the dot products by 1/√dk

This normalization allows the attention mechanism to learn flexible, interpretable relationships between different positions in the input sequence, with each posit

In [ ]:
# The answer text.
prog_answer = "".join(b.get("text", "") for b in result.message["content"] if "text" in b)
print(prog_answer[:300], "...")

Based on the "Attention Is All You Need" paper [1706.03762], attention scores sum to 1.0 due to the softmax normalization. 

Specifically, in the Scaled Dot-Product Attention formula (Equation 1), the softmax function is used to convert raw compatibility scores into a probability distribution:

Atte ...


In [ ]:
# Which document(s) did the agent actually open? Scan its transcript.
doc_ids = []
for m in agent.messages:
    for b in m.get("content", []):
        if isinstance(b, dict) and "toolUse" in b and b["toolUse"]["name"] == "load_document":
            doc_ids.append(b["toolUse"]["input"]["doc_id"])
doc_loaded = doc_ids[0] if doc_ids else "NONE"
doc_loaded

'1706.03762'

In [ ]:
# The REAL tokens the agent used across the whole loop.
prog_tokens = result.metrics.accumulated_usage["inputTokens"]
print(f"Progressive input tokens: {prog_tokens:,}")

Progressive input tokens: 25,104


## Compare the two arms

In [ ]:
import pandas as pd
pd.DataFrame([
    {"Arm": "Naive (all 7)",   "Input tokens": naive_input_tokens, "Latency": round(naive_latency, 2), "Doc": "ALL 7",     "Cited": "no"},
    {"Arm": "Progressive",     "Input tokens": prog_tokens,        "Latency": round(prog_latency, 2),  "Doc": doc_loaded,  "Cited": "yes"},
])

,Arm,Input tokens,Latency,Doc,Cited
0,Naive (all 7),200250,55.05,ALL 7,no
1,Progressive,25104,7.70,1706.03762,yes


In [ ]:
print(f"Token reduction: {naive_input_tokens / prog_tokens:.1f}x fewer tokens")

Token reduction: 8.0x fewer tokens


In [ ]:
# Token X-ray: where did every token go?
def xray(label, breakdown):
    total = sum(breakdown.values())
    print(f"\n{label} - {total:,} tokens")
    for src, tok in breakdown.items():
        pct = tok / total * 100 if total else 0
        print(f"  {src:<28} {'#' * int(pct/2):<50} {tok:>7,} ({pct:.0f}%)")

In [ ]:
xray("Naive", naive_breakdown)
prog_breakdown = {
    "system+index": count_tokens(system_prompt),
    "document": count_tokens((PROJECT_ROOT/'data'/'corpus'/f'{doc_loaded}.txt').read_text(errors='ignore')) if doc_loaded != "NONE" else 0,
    "question": count_tokens(DEMO_QUERY),
}
xray("Progressive", prog_breakdown)


Naive - 144,187 tokens
  Attention Is All You Need    ##                                                   7,924 (5%)
  Lost in the Middle           ####                                                12,696 (9%)
  RULER                        #####                                               17,156 (12%)
  Retrieval-Augmented Generation ####                                                12,852 (9%)
  A Survey on Hallucination in LLMs ################                                    47,554 (33%)
  Indirect Prompt Injection    ########                                            23,412 (16%)
  RLHF effects on diversity    #######                                             22,593 (16%)

Progressive - 8,119 tokens
  system+index                 #                                                      185 (2%)
  document                     ################################################     7,924 (98%)
  question                                                                       

## What this proved

1. **Attention budget is finite.** All 7 papers = mostly irrelevant tokens. Attention thins.
2. **Progressive disclosure stays lean.** Start with the tiny index, add only the one paper needed.
3. **References, not full data (Sec 7.5).** The index is the map; `load_document` fetches the territory, only when needed.

**Next:** Experiment 2 - how context *grows* across agent-loop turns, and how budgeting stops the rot.